# Overlap Group-Rep Signature Similarity

This notebook starts from the overlap-filtered `.h5ad` files produced by `scripts/build_overlap_filtered_h5ads.py` to determine the retained lines and matched grouped-replicate samples, then loads the corresponding per-line DE files from the source result directories.

For each dataset pair, it reconstructs matches using the same rule as the overlap script / heatmap notebook:

- same `pubchem_cid`
- same `cell_type`
- same `pert_time_h`
- mutual nearest neighbors in `log10(pert_dose_uM)`
- `|Δ log10 dose| <= 1`
- a `cell_type + pert_time_h` context is included for a dataset pair only if that pair shares at least `10` matched drugs in that context

For every matched grouped-replicate sample pair, the notebook computes:

- signed Spearman correlation across genes on `logFC`
- signed Spearman correlation across genes on `t`
- signed overlap@50 on `t`, using the top 50 and bottom 50 genes

All signature metrics are computed twice:

- on the usual pairwise shared gene set for that dataset pair / line comparison
- on the line-specific multi-dataset gene set for that `cell_type`, defined as the genes shared across all active datasets that retain that line, exposed via `*_global` columns in the saved tables

It also computes a baseline for each matched sample: the average signature across other non-control compounds from the same dataset, line, dose, and time. Baseline metrics are evaluated on the same gene space as the corresponding observed metric, both for the pairwise shared-gene evaluation and for the line-specific multi-dataset shared-gene evaluation.

For summary tables, matched sample pairs are first averaged within each `pubchem_cid + cell_type + pert_time_h` case across all matched doses, then those drug-line-time cases are averaged to get the dataset-pair and dataset-pair-line summaries.

Outputs are written to `results/overlap_signature_similarity_group_rep/`.


In [ ]:
from __future__ import annotations

from collections import defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional
import itertools
import os

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy.stats import rankdata


sns.set_theme(style="whitegrid")


In [ ]:
DATASET_ORDER = [
    "l1000_phase1",
    "l1000_phase2",
    "tahoe",
    "cigs_mce",
    "novartis_batch_2500",
    "vcpi_0001",
    "cigs_tcm",
    "vcpi_0002",
    "gdpx2",
    "sciplex",
    "dilimap_train_val",
    "op3",
]
DISPLAY_LABELS = {
    "l1000_phase1": "L1000 Phase I",
    "l1000_phase2": "L1000 Phase II",
    "tahoe": "Tahoe-100M",
    "cigs_mce": "CIGS-MCE",
    "novartis_batch_2500": "Novartis/DRUG-seq U2OS",
    "vcpi_0001": "VCPI-0001",
    "cigs_tcm": "CIGS-TCM",
    "vcpi_0002": "VCPI-0002",
    "gdpx2": "GDPx2",
    "sciplex": "sci-Plex",
    "dilimap_train_val": "DILImap",
    "op3": "OP3",
}
DATA_ROOT = Path(os.environ.get("PERTURB_DATA_ROOT", "../data/processed"))


def group_rep_results(dataset_name: str, min_cells: int) -> Path:
    return (
        DATA_ROOT
        / dataset_name
        / "deg_data"
        / "group_rep"
        / "full"
        / "qc_false"
        / f"filter_min_cells_{min_cells}"
        / "results"
    )


SOURCE_DATASET_DIRS = {
    "l1000_phase1": group_rep_results("l1000_phase1", 0),
    "l1000_phase2": group_rep_results("l1000_phase2", 0),
    "tahoe": group_rep_results("tahoe", 50),
    "cigs_mce": group_rep_results("cigs_mce", 0),
    "novartis_batch_2500": group_rep_results("novartis_batch_2500", 0),
    "vcpi_0001": group_rep_results("vcpi_0001", 0),
    "cigs_tcm": group_rep_results("cigs_tcm", 0),
    "vcpi_0002": group_rep_results("vcpi_0002", 0),
    "gdpx2": group_rep_results("gdpx2", 0),
    "sciplex": group_rep_results("sciplex", 10),
    "dilimap_train_val": group_rep_results("dilimap_train_val", 0),
    "op3": group_rep_results("op3", 10),
}
MAX_LOG10_DOSE_DIFF = 1.0
MIN_CONTEXT_SHARED_DRUGS = 10
TOP_K = 50
NUMERIC_SIG_FIGS = 12
INVALID_STRING_VALUES = {"", "nan", "none", "<na>"}


def find_repo_root(start=None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "results" / "overlap_filtered_h5ads").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")


REPO_ROOT = find_repo_root()
OVERLAP_DIR = REPO_ROOT / "results" / "overlap_filtered_h5ads"
OUTPUT_DIR = REPO_ROOT / "results" / "overlap_signature_similarity_group_rep"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {REPO_ROOT}")
print(f"Overlap directory: {OVERLAP_DIR}")
print(f"Output directory: {OUTPUT_DIR}")


In [ ]:
def pretty_label(dataset_name: str) -> str:
    return DISPLAY_LABELS.get(dataset_name, dataset_name)


def format_numeric(value: float) -> str:
    formatted = f"{value:.{NUMERIC_SIG_FIGS}g}"
    return "0" if formatted == "-0" else formatted


def sanitize_string_values(series: pd.Series) -> pd.Series:
    normalized = series.astype("string").fillna("").astype(str).str.strip()
    normalized.loc[normalized.str.lower().isin(INVALID_STRING_VALUES)] = ""
    return normalized


def format_pubchem_cid(value: float) -> str:
    if float(value).is_integer():
        return str(int(value))
    return format_numeric(float(value))


def normalize_pubchem_cid_values(series: pd.Series) -> pd.Series:
    normalized = sanitize_string_values(series)
    numeric = pd.to_numeric(normalized, errors="coerce")
    finite_mask = np.isfinite(numeric.to_numpy(dtype=float))
    if not finite_mask.any():
        return normalized

    normalized = normalized.copy()
    normalized.loc[finite_mask] = numeric.loc[finite_mask].map(format_pubchem_cid)
    return normalized


def coerce_control_mask(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False).astype(bool)
    normalized = values.astype("string").fillna("").astype(str).str.strip().str.lower()
    return normalized.isin({"true", "1", "yes"})


EMPTY_OVERLAP_FRAME = pd.DataFrame(
    columns=[
        "dataset_name",
        "obs_id",
        "plate",
        "well",
        "pubchem_cid",
        "cell_type",
        "pert_time_h",
        "pert_dose_uM",
        "time_key",
        "dose_key",
        "log10_dose",
    ]
)


def load_overlap_obs(dataset_name: str, overlap_dir: Path = OVERLAP_DIR) -> pd.DataFrame:
    h5ad_path = overlap_dir / f"{dataset_name}_overlap_filtered.h5ad"
    if not h5ad_path.exists():
        raise FileNotFoundError(f"Missing overlap file: {h5ad_path}")

    adata = ad.read_h5ad(h5ad_path, backed="r")
    try:
        obs = adata.obs.copy()
    finally:
        adata.file.close()

    if obs.empty:
        return EMPTY_OVERLAP_FRAME.copy()
    if "is_control" not in obs.columns:
        raise KeyError(f"{h5ad_path} is missing obs['is_control']")

    obs = obs.loc[~coerce_control_mask(obs["is_control"])].copy()
    if obs.empty:
        return EMPTY_OVERLAP_FRAME.copy()

    frame = pd.DataFrame(index=obs.index.copy())
    frame["dataset_name"] = dataset_name
    frame["obs_id"] = frame.index.astype(str)
    frame["plate"] = (
        obs["plate"].astype("string").fillna("").astype(str).str.strip()
        if "plate" in obs.columns
        else ""
    )
    frame["well"] = (
        obs["well"].astype("string").fillna("").astype(str).str.strip()
        if "well" in obs.columns
        else ""
    )
    context_column = "harmonized_context_key" if "harmonized_context_key" in obs.columns else "cell_type"
    frame["pubchem_cid"] = normalize_pubchem_cid_values(obs["pubchem_cid"])
    frame["cell_type"] = sanitize_string_values(obs[context_column])
    frame["pert_time_h"] = pd.to_numeric(obs["pert_time_h"], errors="coerce")
    frame["pert_dose_uM"] = pd.to_numeric(obs["pert_dose_uM"], errors="coerce")

    valid_mask = (
        (frame["pubchem_cid"] != "")
        & (frame["cell_type"] != "")
        & np.isfinite(frame["pert_time_h"].to_numpy(dtype=float))
        & np.isfinite(frame["pert_dose_uM"].to_numpy(dtype=float))
        & (frame["pert_dose_uM"].to_numpy(dtype=float) > 0.0)
    )
    frame = frame.loc[valid_mask].copy().reset_index(drop=True)
    if frame.empty:
        return EMPTY_OVERLAP_FRAME.copy()

    frame["time_key"] = frame["pert_time_h"].map(lambda value: format_numeric(float(value)))
    frame["dose_key"] = frame["pert_dose_uM"].map(lambda value: format_numeric(float(value)))
    frame["log10_dose"] = np.log10(frame["pert_dose_uM"].to_numpy(dtype=np.float64))
    return frame[EMPTY_OVERLAP_FRAME.columns.tolist()]


def ensure_overlap_frame_schema(frame: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    if frame is None or frame.empty:
        return EMPTY_OVERLAP_FRAME.copy()

    frame = frame.copy()
    if "dataset_name" not in frame.columns:
        frame["dataset_name"] = dataset_name
    else:
        frame["dataset_name"] = frame["dataset_name"].astype("string").fillna(dataset_name).astype(str).str.strip()

    if "obs_id" not in frame.columns:
        frame["obs_id"] = pd.Index(frame.index).astype(str)
    else:
        frame["obs_id"] = frame["obs_id"].astype("string").fillna("").astype(str).str.strip()

    for column_name in ["plate", "well"]:
        if column_name not in frame.columns:
            frame[column_name] = ""
        else:
            frame[column_name] = frame[column_name].astype("string").fillna("").astype(str).str.strip()

    required_columns = ["pubchem_cid", "cell_type", "pert_time_h", "pert_dose_uM"]
    missing_columns = [column_name for column_name in required_columns if column_name not in frame.columns]
    if missing_columns:
        raise KeyError(
            f"Overlap frame for {dataset_name} is missing required columns: {missing_columns}"
        )

    frame["pubchem_cid"] = normalize_pubchem_cid_values(frame["pubchem_cid"])
    frame["cell_type"] = sanitize_string_values(frame["cell_type"])
    frame["pert_time_h"] = pd.to_numeric(frame["pert_time_h"], errors="coerce")
    frame["pert_dose_uM"] = pd.to_numeric(frame["pert_dose_uM"], errors="coerce")

    valid_mask = (
        (frame["pubchem_cid"] != "")
        & (frame["cell_type"] != "")
        & np.isfinite(frame["pert_time_h"].to_numpy(dtype=float))
        & np.isfinite(frame["pert_dose_uM"].to_numpy(dtype=float))
        & (frame["pert_dose_uM"].to_numpy(dtype=float) > 0.0)
    )
    frame = frame.loc[valid_mask].copy().reset_index(drop=True)
    if frame.empty:
        return EMPTY_OVERLAP_FRAME.copy()

    frame["time_key"] = frame["pert_time_h"].map(lambda value: format_numeric(float(value)))
    frame["dose_key"] = frame["pert_dose_uM"].map(lambda value: format_numeric(float(value)))
    frame["log10_dose"] = np.log10(frame["pert_dose_uM"].to_numpy(dtype=np.float64))
    return frame[EMPTY_OVERLAP_FRAME.columns.tolist()]


def build_groups(frame: pd.DataFrame) -> dict[tuple[str, str, str], np.ndarray]:
    if frame.empty:
        return {}
    grouped = frame.groupby(["pubchem_cid", "cell_type", "time_key"], sort=False).groups
    return {
        key: np.asarray(list(row_positions), dtype=np.int64)
        for key, row_positions in grouped.items()
    }


def build_dataset_index(dataset_name: str) -> dict[str, object]:
    frame = ensure_overlap_frame_schema(load_overlap_obs(dataset_name), dataset_name)
    return {
        "frame": frame,
        "groups": build_groups(frame),
    }


def active_dataset_names(dataset_indices: dict[str, dict[str, object]]) -> list[str]:
    return [
        dataset_name
        for dataset_name in DATASET_ORDER
        if dataset_name in dataset_indices and not dataset_indices[dataset_name]["frame"].empty
    ]


def mutual_nearest_logdose_pairs(
    left_log10_doses: np.ndarray,
    right_log10_doses: np.ndarray,
    max_log10_dose_diff: float = MAX_LOG10_DOSE_DIFF,
) -> np.ndarray:
    if left_log10_doses.size == 0 or right_log10_doses.size == 0:
        return np.empty((0, 2), dtype=np.int64)

    diff = np.abs(left_log10_doses[:, None] - right_log10_doses[None, :])
    left_min = diff.min(axis=1, keepdims=True)
    right_min = diff.min(axis=0, keepdims=True)
    is_mnn = (
        (diff <= max_log10_dose_diff + 1e-12)
        & np.isclose(diff, left_min, rtol=0.0, atol=1e-12)
        & np.isclose(diff, right_min, rtol=0.0, atol=1e-12)
    )
    return np.argwhere(is_mnn)


MATCH_PAIR_COLUMNS = [
    "dataset_a",
    "dataset_b",
    "cell_type",
    "pubchem_cid",
    "time_key",
    "left_obs_id",
    "right_obs_id",
    "left_plate",
    "right_plate",
    "left_well",
    "right_well",
    "left_dose_key",
    "right_dose_key",
    "left_log10_dose",
    "right_log10_dose",
    "abs_delta_log10_dose",
    "matched_condition_key",
    "n_context_matching_drugs",
]


def pair_match_frame(
    left_dataset: str,
    right_dataset: str,
    left_index: dict[str, object],
    right_index: dict[str, object],
    *,
    max_log10_dose_diff: float = MAX_LOG10_DOSE_DIFF,
    min_context_shared_drugs: int = MIN_CONTEXT_SHARED_DRUGS,
) -> pd.DataFrame:
    left_frame = ensure_overlap_frame_schema(left_index["frame"], left_dataset)
    right_frame = ensure_overlap_frame_schema(right_index["frame"], right_dataset)
    left_groups = left_index["groups"]
    right_groups = right_index["groups"]

    if set(build_groups(left_frame)) != set(left_groups):
        left_groups = build_groups(left_frame)
    if set(build_groups(right_frame)) != set(right_groups):
        right_groups = build_groups(right_frame)

    if left_frame.empty or right_frame.empty:
        return pd.DataFrame(columns=MATCH_PAIR_COLUMNS)

    shared_keys = sorted(set(left_groups) & set(right_groups))
    if not shared_keys:
        return pd.DataFrame(columns=MATCH_PAIR_COLUMNS)

    left_obs_ids = left_frame["obs_id"].to_numpy(dtype=object)
    right_obs_ids = right_frame["obs_id"].to_numpy(dtype=object)
    left_plates = left_frame["plate"].to_numpy(dtype=object)
    right_plates = right_frame["plate"].to_numpy(dtype=object)
    left_wells = left_frame["well"].to_numpy(dtype=object)
    right_wells = right_frame["well"].to_numpy(dtype=object)
    left_log10_dose = left_frame["log10_dose"].to_numpy(dtype=np.float64)
    right_log10_dose = right_frame["log10_dose"].to_numpy(dtype=np.float64)
    left_dose_keys = left_frame["dose_key"].to_numpy(dtype=object)
    right_dose_keys = right_frame["dose_key"].to_numpy(dtype=object)

    context_matching_drugs: dict[tuple[str, str], set[str]] = defaultdict(set)
    rows: list[dict[str, object]] = []

    for pubchem_cid, cell_type, time_key in shared_keys:
        left_rows = left_groups[(pubchem_cid, cell_type, time_key)]
        right_rows = right_groups[(pubchem_cid, cell_type, time_key)]
        pairs = mutual_nearest_logdose_pairs(
            left_log10_doses=left_log10_dose[left_rows],
            right_log10_doses=right_log10_dose[right_rows],
            max_log10_dose_diff=max_log10_dose_diff,
        )
        if pairs.size == 0:
            continue

        for left_pos, right_pos in pairs:
            left_row = int(left_rows[left_pos])
            right_row = int(right_rows[right_pos])
            left_dose_key = str(left_dose_keys[left_row])
            right_dose_key = str(right_dose_keys[right_row])
            context_matching_drugs[(str(cell_type), str(time_key))].add(str(pubchem_cid))
            rows.append(
                {
                    "dataset_a": left_dataset,
                    "dataset_b": right_dataset,
                    "cell_type": str(cell_type),
                    "pubchem_cid": str(pubchem_cid),
                    "time_key": str(time_key),
                    "left_obs_id": str(left_obs_ids[left_row]),
                    "right_obs_id": str(right_obs_ids[right_row]),
                    "left_plate": str(left_plates[left_row]),
                    "right_plate": str(right_plates[right_row]),
                    "left_well": str(left_wells[left_row]),
                    "right_well": str(right_wells[right_row]),
                    "left_dose_key": left_dose_key,
                    "right_dose_key": right_dose_key,
                    "left_log10_dose": float(left_log10_dose[left_row]),
                    "right_log10_dose": float(right_log10_dose[right_row]),
                    "abs_delta_log10_dose": float(abs(left_log10_dose[left_row] - right_log10_dose[right_row])),
                    "matched_condition_key": "|".join(
                        [
                            str(pubchem_cid),
                            str(cell_type),
                            str(time_key),
                            left_dose_key,
                            right_dose_key,
                        ]
                    ),
                }
            )

    if not rows:
        return pd.DataFrame(columns=MATCH_PAIR_COLUMNS)

    frame = pd.DataFrame(rows)
    qualifying_context_drug_counts = {
        context_key: len(compounds)
        for context_key, compounds in context_matching_drugs.items()
        if len(compounds) >= int(min_context_shared_drugs)
    }
    if not qualifying_context_drug_counts:
        return pd.DataFrame(columns=MATCH_PAIR_COLUMNS)

    frame["_context_key"] = list(zip(frame["cell_type"], frame["time_key"]))
    frame["n_context_matching_drugs"] = frame["_context_key"].map(qualifying_context_drug_counts)
    frame = frame.loc[frame["n_context_matching_drugs"].notna()].copy()
    frame["n_context_matching_drugs"] = frame["n_context_matching_drugs"].astype(int)
    frame = frame.drop(columns="_context_key")
    return frame[MATCH_PAIR_COLUMNS].reset_index(drop=True)



In [ ]:
@dataclass
class LineSource:
    dataset_name: str
    cell_type: str
    path: Path
    adata: ad.AnnData = field(init=False, repr=False)
    obs: pd.DataFrame = field(init=False, repr=False)
    unique_gene_keys: np.ndarray = field(init=False, repr=False)
    unique_gene_positions: np.ndarray = field(init=False, repr=False)
    gene_to_pos: dict[str, int] = field(init=False, repr=False)
    lookup_row_pos: dict[str, int] = field(init=False, repr=False)
    _vector_cache: dict[tuple[str, int], np.ndarray] = field(default_factory=dict, init=False, repr=False)
    _baseline_cache: dict[tuple[str, int], object] = field(default_factory=dict, init=False, repr=False)
    _baseline_peer_counts: dict[int, int] = field(default_factory=dict, init=False, repr=False)

    def __post_init__(self) -> None:
        self.adata = ad.read_h5ad(self.path, backed="r")
        obs = self.adata.obs.copy()
        row_positions = np.arange(self.adata.n_obs, dtype=np.int64)

        if "is_control" not in obs.columns:
            raise KeyError(f"{self.path} is missing obs['is_control']")

        obs["source_index"] = obs.index.astype(str)
        control_mask = coerce_control_mask(obs["is_control"]).to_numpy(dtype=bool)
        obs = obs.loc[~control_mask].copy()
        row_positions = row_positions[~control_mask]

        obs["source_row_pos"] = row_positions
        for column_name in [
            "id",
            "plate",
            "well",
            "cell_type",
            "perturbagen",
            "perturbagen_name",
            "perturbation_label",
            "pubchem_cid",
        ]:
            if column_name in obs.columns:
                obs[column_name] = obs[column_name].astype("string").fillna("").astype(str).str.strip()

        if "pubchem_cid" in obs.columns:
            obs["pubchem_cid"] = normalize_pubchem_cid_values(obs["pubchem_cid"])

        obs["pert_time_h"] = pd.to_numeric(obs["pert_time_h"], errors="coerce")
        obs["pert_dose_uM"] = pd.to_numeric(obs["pert_dose_uM"], errors="coerce")
        obs["time_key"] = obs["pert_time_h"].map(
            lambda value: format_numeric(float(value)) if pd.notna(value) else ""
        )
        obs["dose_key"] = obs["pert_dose_uM"].map(
            lambda value: format_numeric(float(value)) if pd.notna(value) and float(value) > 0 else ""
        )

        valid_mask = (
            (obs["pubchem_cid"] != "")
            & np.isfinite(obs["pert_time_h"].to_numpy(dtype=float))
            & np.isfinite(obs["pert_dose_uM"].to_numpy(dtype=float))
            & (obs["pert_dose_uM"].to_numpy(dtype=float) > 0.0)
        )
        obs = obs.loc[valid_mask].copy()
        obs = obs.set_index("source_row_pos", drop=False)
        self.obs = obs
        self.lookup_row_pos = self._build_lookup_row_pos()

        var = self.adata.var.copy()
        if "symbol" in var.columns:
            gene_key_series = pd.Series(
                var["symbol"].astype("string").fillna("").astype(str).str.strip().to_numpy(),
                index=np.arange(self.adata.n_vars, dtype=np.int64),
            )
        else:
            gene_key_series = pd.Series(
                pd.Index(self.adata.var_names.astype(str)).astype(str).str.strip().to_numpy(),
                index=np.arange(self.adata.n_vars, dtype=np.int64),
            )
        keep_mask = (gene_key_series != "") & ~gene_key_series.duplicated(keep="first")
        self.unique_gene_positions = gene_key_series.index[keep_mask].to_numpy(dtype=np.int64)
        self.unique_gene_keys = gene_key_series.loc[keep_mask].to_numpy(dtype=object)
        self.gene_to_pos = {
            str(gene_key): int(pos)
            for pos, gene_key in enumerate(self.unique_gene_keys.tolist())
        }

    def _build_lookup_row_pos(self) -> dict[str, int]:
        lookup_row_pos: dict[str, int] = {}

        def add_lookup_key(key: str, row_pos: int) -> None:
            if not key or key.lower() == "nan":
                return
            lookup_row_pos.setdefault(key, row_pos)

        for row_pos, row in self.obs.iterrows():
            row_pos = int(row_pos)
            for column_name in ["source_index", "id"]:
                if column_name in row.index:
                    add_lookup_key(str(row[column_name]).strip(), row_pos)

            cell_type = str(row.get("cell_type", "")).strip()
            pubchem_cid = str(row.get("pubchem_cid", "")).strip()
            dose_key = str(row.get("dose_key", "")).strip()
            time_key = str(row.get("time_key", "")).strip()

            if pubchem_cid and dose_key and time_key and cell_type:
                add_lookup_key(
                    f"{pubchem_cid}|{dose_key}|{time_key}|{cell_type}",
                    row_pos,
                )
        return lookup_row_pos

    def resolve_row_pos(
        self,
        obs_id: str,
        *,
        pubchem_cid=None,
        dose_key=None,
        time_key=None,
        plate=None,
        well=None,
    ) -> int:
        obs_id = str(obs_id)
        candidate_keys = [obs_id]

        pubchem_cid = "" if pubchem_cid is None else str(pubchem_cid).strip()
        dose_key = "" if dose_key is None else str(dose_key).strip()
        time_key = "" if time_key is None else str(time_key).strip()

        if pubchem_cid and dose_key and time_key:
            candidate_keys.append(f"{pubchem_cid}|{dose_key}|{time_key}|{self.cell_type}")

        for candidate_key in candidate_keys:
            if candidate_key in self.lookup_row_pos:
                return int(self.lookup_row_pos[candidate_key])

        raise KeyError(
            f"Could not resolve obs_id={obs_id!r} in {self.path}. "
            f"Tried {candidate_keys!r}. Available lookup keys: {len(self.lookup_row_pos):,}"
        )

    def get_vector(
        self,
        obs_id: str,
        layer_name: str,
        *,
        pubchem_cid=None,
        dose_key=None,
        time_key=None,
        plate=None,
        well=None,
    ) -> np.ndarray:
        row_pos = self.resolve_row_pos(
            obs_id,
            pubchem_cid=pubchem_cid,
            dose_key=dose_key,
            time_key=time_key,
            plate=plate,
            well=well,
        )
        cache_key = (layer_name, row_pos)
        if cache_key not in self._vector_cache:
            vector = np.asarray(self.adata.layers[layer_name][row_pos], dtype=np.float32).reshape(-1)
            self._vector_cache[cache_key] = vector[self.unique_gene_positions]
        return self._vector_cache[cache_key]

    def get_baseline_vector(
        self,
        obs_id: str,
        layer_name: str,
        *,
        pubchem_cid=None,
        dose_key=None,
        time_key=None,
        plate=None,
        well=None,
    ):
        row_pos = self.resolve_row_pos(
            obs_id,
            pubchem_cid=pubchem_cid,
            dose_key=dose_key,
            time_key=time_key,
            plate=plate,
            well=well,
        )
        cache_key = (layer_name, row_pos)
        if cache_key not in self._baseline_cache:
            row = self.obs.loc[row_pos]
            peer_obs = self.obs.loc[
                (self.obs["dose_key"] == row["dose_key"])
                & (self.obs["time_key"] == row["time_key"])
                & (self.obs["pubchem_cid"] != row["pubchem_cid"])
            ]
            peer_rows = peer_obs["source_row_pos"].to_numpy(dtype=np.int64)
            self._baseline_peer_counts[row_pos] = int(len(peer_rows))
            if len(peer_rows) == 0:
                self._baseline_cache[cache_key] = None
            else:
                matrix = np.asarray(self.adata.layers[layer_name][peer_rows], dtype=np.float32)
                if matrix.ndim == 1:
                    matrix = matrix[np.newaxis, :]
                baseline = matrix[:, self.unique_gene_positions].mean(axis=0, dtype=np.float64)
                self._baseline_cache[cache_key] = np.asarray(baseline, dtype=np.float32)
        return self._baseline_cache[cache_key]

    def baseline_peer_count(
        self,
        obs_id: str,
        *,
        pubchem_cid=None,
        dose_key=None,
        time_key=None,
        plate=None,
        well=None,
    ) -> int:
        row_pos = self.resolve_row_pos(
            obs_id,
            pubchem_cid=pubchem_cid,
            dose_key=dose_key,
            time_key=time_key,
            plate=plate,
            well=well,
        )
        if row_pos not in self._baseline_peer_counts:
            _ = self.get_baseline_vector(
                obs_id,
                "logFC",
                pubchem_cid=pubchem_cid,
                dose_key=dose_key,
                time_key=time_key,
                plate=plate,
                well=well,
            )
        return int(self._baseline_peer_counts.get(row_pos, 0))

    def close(self) -> None:
        self.adata.file.close()


def resolve_line_path(dataset_name: str, cell_type: str) -> Path:
    dataset_dir = SOURCE_DATASET_DIRS[dataset_name]
    candidates = [
        dataset_dir / f"{cell_type}_de.h5ad",
        dataset_dir / f"{cell_type}.h5ad",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find a line file for dataset={dataset_name}, cell_type={cell_type} in {dataset_dir}"
    )


LINE_SOURCE_CACHE: dict[tuple[str, str], LineSource] = {}
COMMON_GENE_CACHE: dict[tuple[str, str, str], tuple[np.ndarray, np.ndarray, np.ndarray]] = {}


def get_line_source(dataset_name: str, cell_type: str) -> LineSource:
    cache_key = (dataset_name, cell_type)
    if cache_key not in LINE_SOURCE_CACHE:
        LINE_SOURCE_CACHE[cache_key] = LineSource(
            dataset_name=dataset_name,
            cell_type=cell_type,
            path=resolve_line_path(dataset_name, cell_type),
        )
    return LINE_SOURCE_CACHE[cache_key]


def shared_gene_positions(
    left_source: LineSource,
    right_source: LineSource,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    cache_key = (left_source.dataset_name, right_source.dataset_name, left_source.cell_type)
    if cache_key not in COMMON_GENE_CACHE:
        shared_genes = [
            gene_key
            for gene_key in left_source.unique_gene_keys.tolist()
            if str(gene_key) in right_source.gene_to_pos
        ]
        left_positions = np.fromiter(
            (left_source.gene_to_pos[str(gene_key)] for gene_key in shared_genes),
            dtype=np.int64,
            count=len(shared_genes),
        )
        right_positions = np.fromiter(
            (right_source.gene_to_pos[str(gene_key)] for gene_key in shared_genes),
            dtype=np.int64,
            count=len(shared_genes),
        )
        COMMON_GENE_CACHE[cache_key] = (
            np.asarray(shared_genes, dtype=object),
            left_positions,
            right_positions,
        )
    return COMMON_GENE_CACHE[cache_key]

LINE_GLOBAL_SHARED_GENE_KEYS: dict[str, np.ndarray] = {}
GLOBAL_GENE_POSITION_CACHE: dict[tuple[str, str], np.ndarray] = {}


def set_global_shared_gene_keys(
    retained_lines: dict[str, list[str]],
    dataset_names: list[str],
) -> dict[str, np.ndarray]:
    global LINE_GLOBAL_SHARED_GENE_KEYS, GLOBAL_GENE_POSITION_CACHE

    cell_types = sorted(
        {
            cell_type
            for dataset_name in dataset_names
            for cell_type in retained_lines.get(dataset_name, [])
        }
    )
    line_gene_map: dict[str, np.ndarray] = {}
    for cell_type in cell_types:
        gene_sets: list[set[str]] = []
        for dataset_name in dataset_names:
            if cell_type not in retained_lines.get(dataset_name, []):
                continue
            source = get_line_source(dataset_name, cell_type)
            gene_sets.append({str(gene_key) for gene_key in source.unique_gene_keys.tolist()})
        if not gene_sets:
            line_gene_map[cell_type] = np.empty(0, dtype=object)
        else:
            line_gene_map[cell_type] = np.asarray(sorted(set.intersection(*gene_sets)), dtype=object)

    LINE_GLOBAL_SHARED_GENE_KEYS = line_gene_map
    GLOBAL_GENE_POSITION_CACHE = {}
    return LINE_GLOBAL_SHARED_GENE_KEYS


def global_gene_positions(source: LineSource) -> tuple[np.ndarray, np.ndarray]:
    cache_key = (source.dataset_name, source.cell_type)
    line_gene_keys = LINE_GLOBAL_SHARED_GENE_KEYS.get(source.cell_type, np.empty(0, dtype=object))
    if line_gene_keys.size == 0:
        return line_gene_keys, np.empty(0, dtype=np.int64)
    if cache_key not in GLOBAL_GENE_POSITION_CACHE:
        missing_gene_keys = [
            str(gene_key)
            for gene_key in line_gene_keys.tolist()
            if str(gene_key) not in source.gene_to_pos
        ]
        if missing_gene_keys:
            raise KeyError(
                f"Line-specific shared-gene set is inconsistent for {(source.dataset_name, source.cell_type)}; "
                f"missing {len(missing_gene_keys)} genes."
            )
        GLOBAL_GENE_POSITION_CACHE[cache_key] = np.fromiter(
            (source.gene_to_pos[str(gene_key)] for gene_key in line_gene_keys.tolist()),
            dtype=np.int64,
            count=int(line_gene_keys.size),
        )
    return line_gene_keys, GLOBAL_GENE_POSITION_CACHE[cache_key]



def filter_finite_pair(left_values: np.ndarray, right_values: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    mask = np.isfinite(left_values) & np.isfinite(right_values)
    return left_values[mask], right_values[mask]


def signed_spearman(left_values: np.ndarray, right_values: np.ndarray) -> float:
    left_values, right_values = filter_finite_pair(
        np.asarray(left_values, dtype=np.float64),
        np.asarray(right_values, dtype=np.float64),
    )
    if left_values.size < 2:
        return float("nan")

    left_ranks = rankdata(left_values, method="average")
    right_ranks = rankdata(right_values, method="average")
    if np.allclose(left_ranks, left_ranks[0]) or np.allclose(right_ranks, right_ranks[0]):
        return float("nan")
    return float(np.corrcoef(left_ranks, right_ranks)[0, 1])


def signed_overlap_at_k(left_values: np.ndarray, right_values: np.ndarray, *, k: int = TOP_K) -> float:
    left_values, right_values = filter_finite_pair(
        np.asarray(left_values, dtype=np.float64),
        np.asarray(right_values, dtype=np.float64),
    )
    n_genes = int(left_values.size)
    k_eff = min(int(k), n_genes // 2)
    if k_eff < 1:
        return float("nan")

    top_left = np.argpartition(left_values, -k_eff)[-k_eff:]
    top_right = np.argpartition(right_values, -k_eff)[-k_eff:]
    bottom_left = np.argpartition(left_values, k_eff - 1)[:k_eff]
    bottom_right = np.argpartition(right_values, k_eff - 1)[:k_eff]

    top_overlap = np.intersect1d(top_left, top_right, assume_unique=False).size / k_eff
    bottom_overlap = np.intersect1d(bottom_left, bottom_right, assume_unique=False).size / k_eff
    return float(0.5 * (top_overlap + bottom_overlap))


def score_signature_pair(
    left_logfc: np.ndarray,
    right_logfc: np.ndarray,
    left_t: np.ndarray,
    right_t: np.ndarray,
) -> dict[str, float]:
    return {
        "spearman_logfc": signed_spearman(left_logfc, right_logfc),
        "spearman_t": signed_spearman(left_t, right_t),
        f"signed_overlap_t_top{TOP_K}": signed_overlap_at_k(left_t, right_t, k=TOP_K),
    }


def mean_available(values: list[float]) -> float:
    finite_values = [value for value in values if pd.notna(value)]
    if not finite_values:
        return float("nan")
    return float(np.mean(finite_values))

def empty_score_dict() -> dict[str, float]:
    return {
        "spearman_logfc": float("nan"),
        "spearman_t": float("nan"),
        f"signed_overlap_t_top{TOP_K}": float("nan"),
    }



def compute_metric_record(match_row: pd.Series) -> dict[str, object]:
    left_source = get_line_source(str(match_row["dataset_a"]), str(match_row["cell_type"]))
    right_source = get_line_source(str(match_row["dataset_b"]), str(match_row["cell_type"]))
    shared_genes, left_gene_pos, right_gene_pos = shared_gene_positions(left_source, right_source)
    if shared_genes.size < 2:
        raise ValueError(
            f"Fewer than two shared genes for {match_row['dataset_a']} vs {match_row['dataset_b']} / {match_row['cell_type']}"
        )

    global_shared_genes, left_global_gene_pos = global_gene_positions(left_source)
    _, right_global_gene_pos = global_gene_positions(right_source)

    left_obs_id = str(match_row["left_obs_id"])
    right_obs_id = str(match_row["right_obs_id"])
    pubchem_cid = str(match_row["pubchem_cid"])
    time_key = str(match_row["time_key"])

    left_lookup = {
        "pubchem_cid": pubchem_cid,
        "dose_key": str(match_row["left_dose_key"]),
        "time_key": time_key,
    }
    right_lookup = {
        "pubchem_cid": pubchem_cid,
        "dose_key": str(match_row["right_dose_key"]),
        "time_key": time_key,
    }

    left_logfc_full = left_source.get_vector(left_obs_id, "logFC", **left_lookup)
    right_logfc_full = right_source.get_vector(right_obs_id, "logFC", **right_lookup)
    left_t_full = left_source.get_vector(left_obs_id, "t", **left_lookup)
    right_t_full = right_source.get_vector(right_obs_id, "t", **right_lookup)

    observed_scores = score_signature_pair(
        left_logfc=left_logfc_full[left_gene_pos],
        right_logfc=right_logfc_full[right_gene_pos],
        left_t=left_t_full[left_gene_pos],
        right_t=right_t_full[right_gene_pos],
    )

    observed_scores_global = empty_score_dict()
    if global_shared_genes.size >= 2:
        observed_scores_global = score_signature_pair(
            left_logfc=left_logfc_full[left_global_gene_pos],
            right_logfc=right_logfc_full[right_global_gene_pos],
            left_t=left_t_full[left_global_gene_pos],
            right_t=right_t_full[right_global_gene_pos],
        )

    left_mean_abs_t = mean_available(np.abs(left_t_full[left_gene_pos]).tolist())
    right_mean_abs_t = mean_available(np.abs(right_t_full[right_gene_pos]).tolist())
    pair_mean_abs_t = mean_available([left_mean_abs_t, right_mean_abs_t])

    left_mean_abs_t_global = float("nan")
    right_mean_abs_t_global = float("nan")
    pair_mean_abs_t_global = float("nan")
    if global_shared_genes.size >= 2:
        left_mean_abs_t_global = mean_available(np.abs(left_t_full[left_global_gene_pos]).tolist())
        right_mean_abs_t_global = mean_available(np.abs(right_t_full[right_global_gene_pos]).tolist())
        pair_mean_abs_t_global = mean_available([left_mean_abs_t_global, right_mean_abs_t_global])

    left_baseline_logfc_full = left_source.get_baseline_vector(left_obs_id, "logFC", **left_lookup)
    left_baseline_t_full = left_source.get_baseline_vector(left_obs_id, "t", **left_lookup)
    right_baseline_logfc_full = right_source.get_baseline_vector(right_obs_id, "logFC", **right_lookup)
    right_baseline_t_full = right_source.get_baseline_vector(right_obs_id, "t", **right_lookup)

    left_baseline_scores = empty_score_dict()
    left_baseline_scores_global = empty_score_dict()
    if left_baseline_logfc_full is not None and left_baseline_t_full is not None:
        left_baseline_scores = score_signature_pair(
            left_logfc=left_logfc_full[left_gene_pos],
            right_logfc=left_baseline_logfc_full[left_gene_pos],
            left_t=left_t_full[left_gene_pos],
            right_t=left_baseline_t_full[left_gene_pos],
        )
        if global_shared_genes.size >= 2:
            left_baseline_scores_global = score_signature_pair(
                left_logfc=left_logfc_full[left_global_gene_pos],
                right_logfc=left_baseline_logfc_full[left_global_gene_pos],
                left_t=left_t_full[left_global_gene_pos],
                right_t=left_baseline_t_full[left_global_gene_pos],
            )

    right_baseline_scores = empty_score_dict()
    right_baseline_scores_global = empty_score_dict()
    if right_baseline_logfc_full is not None and right_baseline_t_full is not None:
        right_baseline_scores = score_signature_pair(
            left_logfc=right_logfc_full[right_gene_pos],
            right_logfc=right_baseline_logfc_full[right_gene_pos],
            left_t=right_t_full[right_gene_pos],
            right_t=right_baseline_t_full[right_gene_pos],
        )
        if global_shared_genes.size >= 2:
            right_baseline_scores_global = score_signature_pair(
                left_logfc=right_logfc_full[right_global_gene_pos],
                right_logfc=right_baseline_logfc_full[right_global_gene_pos],
                left_t=right_t_full[right_global_gene_pos],
                right_t=right_baseline_t_full[right_global_gene_pos],
            )

    overlap_column = f"signed_overlap_t_top{TOP_K}"
    return {
        **match_row.to_dict(),
        "n_common_genes": int(shared_genes.size),
        "n_global_common_genes": int(global_shared_genes.size),
        "left_mean_abs_t": left_mean_abs_t,
        "right_mean_abs_t": right_mean_abs_t,
        "pair_mean_abs_t": pair_mean_abs_t,
        "left_mean_abs_t_global": left_mean_abs_t_global,
        "right_mean_abs_t_global": right_mean_abs_t_global,
        "pair_mean_abs_t_global": pair_mean_abs_t_global,
        "observed_spearman_logfc": observed_scores["spearman_logfc"],
        "observed_spearman_logfc_global": observed_scores_global["spearman_logfc"],
        "observed_spearman_t": observed_scores["spearman_t"],
        "observed_spearman_t_global": observed_scores_global["spearman_t"],
        f"observed_{overlap_column}": observed_scores[overlap_column],
        f"observed_{overlap_column}_global": observed_scores_global[overlap_column],
        "left_baseline_peer_count": left_source.baseline_peer_count(left_obs_id, **left_lookup),
        "left_baseline_spearman_logfc": left_baseline_scores["spearman_logfc"],
        "left_baseline_spearman_logfc_global": left_baseline_scores_global["spearman_logfc"],
        "left_baseline_spearman_t": left_baseline_scores["spearman_t"],
        "left_baseline_spearman_t_global": left_baseline_scores_global["spearman_t"],
        f"left_baseline_{overlap_column}": left_baseline_scores[overlap_column],
        f"left_baseline_{overlap_column}_global": left_baseline_scores_global[overlap_column],
        "right_baseline_peer_count": right_source.baseline_peer_count(right_obs_id, **right_lookup),
        "right_baseline_spearman_logfc": right_baseline_scores["spearman_logfc"],
        "right_baseline_spearman_logfc_global": right_baseline_scores_global["spearman_logfc"],
        "right_baseline_spearman_t": right_baseline_scores["spearman_t"],
        "right_baseline_spearman_t_global": right_baseline_scores_global["spearman_t"],
        f"right_baseline_{overlap_column}": right_baseline_scores[overlap_column],
        f"right_baseline_{overlap_column}_global": right_baseline_scores_global[overlap_column],
        "baseline_pair_mean_spearman_logfc": mean_available(
            [left_baseline_scores["spearman_logfc"], right_baseline_scores["spearman_logfc"]]
        ),
        "baseline_pair_mean_spearman_logfc_global": mean_available(
            [left_baseline_scores_global["spearman_logfc"], right_baseline_scores_global["spearman_logfc"]]
        ),
        "baseline_pair_mean_spearman_t": mean_available(
            [left_baseline_scores["spearman_t"], right_baseline_scores["spearman_t"]]
        ),
        "baseline_pair_mean_spearman_t_global": mean_available(
            [left_baseline_scores_global["spearman_t"], right_baseline_scores_global["spearman_t"]]
        ),
        f"baseline_pair_mean_{overlap_column}": mean_available(
            [left_baseline_scores[overlap_column], right_baseline_scores[overlap_column]]
        ),
        f"baseline_pair_mean_{overlap_column}_global": mean_available(
            [left_baseline_scores_global[overlap_column], right_baseline_scores_global[overlap_column]]
        ),
    }

def close_all_line_sources() -> None:
    for line_source in LINE_SOURCE_CACHE.values():
        line_source.close()



In [ ]:
dataset_indices = {dataset_name: build_dataset_index(dataset_name) for dataset_name in DATASET_ORDER}
active_datasets = active_dataset_names(dataset_indices)
if not active_datasets:
    raise ValueError("No overlap-filtered non-control samples were found for the configured datasets.")

print("Datasets in scope:", ", ".join(pretty_label(dataset_name) for dataset_name in active_datasets))

retained_lines = {
    dataset_name: sorted(dataset_indices[dataset_name]["frame"]["cell_type"].unique().tolist())
    for dataset_name in active_datasets
}
retained_lines_display = pd.DataFrame(
    {
        "dataset": [pretty_label(dataset_name) for dataset_name in retained_lines],
        "cell_types": [", ".join(lines) for lines in retained_lines.values()],
    }
)
display(retained_lines_display)

line_global_gene_keys = set_global_shared_gene_keys(retained_lines, active_datasets)
line_global_gene_counts = {
    cell_type: int(gene_keys.size)
    for cell_type, gene_keys in line_global_gene_keys.items()
}
print(f"Line-specific shared-gene sets computed for {len(line_global_gene_counts):,} retained lines.")
if not line_global_gene_counts or max(line_global_gene_counts.values()) < 2:
    print(
        "Line-specific shared-gene evaluation will be unavailable because no retained line has at least two genes shared across the datasets that retain it."
    )
else:
    print(
        f"Line-specific shared-gene count range across retained lines: {min(line_global_gene_counts.values()):,} to {max(line_global_gene_counts.values()):,}"
    )

pair_match_frames: list[pd.DataFrame] = []
for dataset_a, dataset_b in itertools.combinations(active_datasets, 2):
    frame = pair_match_frame(
        left_dataset=dataset_a,
        right_dataset=dataset_b,
        left_index=dataset_indices[dataset_a],
        right_index=dataset_indices[dataset_b],
    )
    if not frame.empty:
        pair_match_frames.append(frame)

if not pair_match_frames:
    raise ValueError("No matched grouped-replicate sample pairs were found.")

matched_pairs = pd.concat(pair_match_frames, ignore_index=True)
matched_pairs_path = OUTPUT_DIR / "matched_sample_pairs.tsv"
matched_pairs.to_csv(matched_pairs_path, sep="\t", index=False)
print(f"Saved matched sample pairs to {matched_pairs_path}")

pair_match_summary = (
    matched_pairs.groupby(["dataset_a", "dataset_b"], as_index=False)
    .agg(
        n_matched_sample_pairs=("left_obs_id", "size"),
        n_matching_drugs=("pubchem_cid", "nunique"),
        n_matching_lines=("cell_type", "nunique"),
        n_matching_conditions=("matched_condition_key", "nunique"),
    )
)
pair_match_summary_display = pair_match_summary.copy()
pair_match_summary_display["dataset_a"] = pair_match_summary_display["dataset_a"].map(pretty_label)
pair_match_summary_display["dataset_b"] = pair_match_summary_display["dataset_b"].map(pretty_label)
display(pair_match_summary_display)

line_match_summary = (
    matched_pairs.groupby(["dataset_a", "dataset_b", "cell_type"], as_index=False)
    .agg(
        n_matched_sample_pairs=("left_obs_id", "size"),
        n_matching_drugs=("pubchem_cid", "nunique"),
        n_matching_conditions=("matched_condition_key", "nunique"),
    )
    .sort_values(["dataset_a", "dataset_b", "cell_type"]) 
    .reset_index(drop=True)
)
line_match_summary_display = line_match_summary.copy()
line_match_summary_display["dataset_a"] = line_match_summary_display["dataset_a"].map(pretty_label)
line_match_summary_display["dataset_b"] = line_match_summary_display["dataset_b"].map(pretty_label)
display(line_match_summary_display)


In [ ]:
metric_records: list[dict[str, object]] = []
unresolved_records: list[dict[str, object]] = []
context_join_columns = [
    "dataset_a",
    "dataset_b",
    "cell_type",
    "time_key",
    "left_dose_key",
    "right_dose_key",
]
for (dataset_a, dataset_b, cell_type, time_key, left_dose_key, right_dose_key), group in matched_pairs.groupby(
    context_join_columns,
    sort=False,
):
    print(
        f"Scoring {pretty_label(dataset_a)} vs {pretty_label(dataset_b)} / {cell_type} / time={time_key} / doses={left_dose_key} vs {right_dose_key}: "
        f"{len(group)} matched sample pairs across {group['pubchem_cid'].nunique()} shared compounds"
    )
    for _, match_row in group.iterrows():
        try:
            metric_records.append(compute_metric_record(match_row))
        except KeyError as exc:
            unresolved_records.append(
                {
                    **match_row.to_dict(),
                    "error": str(exc),
                }
            )

matched_pair_metrics = pd.DataFrame(metric_records)
if matched_pair_metrics.empty:
    raise ValueError("No matched sample pairs could be resolved in the source line files.")

matched_pair_metrics_path = OUTPUT_DIR / "matched_sample_pair_metrics.tsv"
matched_pair_metrics.to_csv(matched_pair_metrics_path, sep="	", index=False)
print(f"Saved matched sample pair metrics to {matched_pair_metrics_path}")
print(f"Scored {len(matched_pair_metrics):,} matched sample pairs")

unresolved_matches = pd.DataFrame(unresolved_records)
if unresolved_matches.empty:
    print("All matched sample pairs were resolved in the source line files.")
else:
    unresolved_matches_path = OUTPUT_DIR / "unresolved_matched_sample_pairs.tsv"
    unresolved_matches.to_csv(unresolved_matches_path, sep="	", index=False)
    print(
        f"Skipped {len(unresolved_matches):,} matched sample pairs with no corresponding grouped result in the source line files. "
        f"Saved details to {unresolved_matches_path}"
    )
    display(
        unresolved_matches.groupby(context_join_columns, as_index=False)
        .agg(n_unresolved=("error", "size"))
        .sort_values(context_join_columns)
        .reset_index(drop=True)
    )

matched_pair_metrics.head()


In [ ]:
overlap_metric_column = f"observed_signed_overlap_t_top{TOP_K}"
baseline_overlap_column = f"baseline_pair_mean_signed_overlap_t_top{TOP_K}"
left_baseline_overlap_column = f"left_baseline_signed_overlap_t_top{TOP_K}"
right_baseline_overlap_column = f"right_baseline_signed_overlap_t_top{TOP_K}"
drug_line_time_group_columns = ["dataset_a", "dataset_b", "cell_type", "time_key", "pubchem_cid"]

metric_mean_columns = {
    "mean_common_genes": "n_common_genes",
    "mean_global_common_genes": "n_global_common_genes",
    "mean_left_mean_abs_t": "left_mean_abs_t",
    "mean_right_mean_abs_t": "right_mean_abs_t",
    "mean_pair_mean_abs_t": "pair_mean_abs_t",
    "mean_left_mean_abs_t_global": "left_mean_abs_t_global",
    "mean_right_mean_abs_t_global": "right_mean_abs_t_global",
    "mean_pair_mean_abs_t_global": "pair_mean_abs_t_global",
    "mean_observed_spearman_logfc": "observed_spearman_logfc",
    "mean_observed_spearman_logfc_global": "observed_spearman_logfc_global",
    "mean_observed_spearman_t": "observed_spearman_t",
    "mean_observed_spearman_t_global": "observed_spearman_t_global",
    f"mean_observed_signed_overlap_t_top{TOP_K}": overlap_metric_column,
    f"mean_observed_signed_overlap_t_top{TOP_K}_global": f"{overlap_metric_column}_global",
    "mean_baseline_pair_spearman_logfc": "baseline_pair_mean_spearman_logfc",
    "mean_baseline_pair_spearman_logfc_global": "baseline_pair_mean_spearman_logfc_global",
    "mean_baseline_pair_spearman_t": "baseline_pair_mean_spearman_t",
    "mean_baseline_pair_spearman_t_global": "baseline_pair_mean_spearman_t_global",
    f"mean_baseline_pair_signed_overlap_t_top{TOP_K}": baseline_overlap_column,
    f"mean_baseline_pair_signed_overlap_t_top{TOP_K}_global": f"{baseline_overlap_column}_global",
    "mean_left_baseline_spearman_logfc": "left_baseline_spearman_logfc",
    "mean_left_baseline_spearman_logfc_global": "left_baseline_spearman_logfc_global",
    "mean_left_baseline_spearman_t": "left_baseline_spearman_t",
    "mean_left_baseline_spearman_t_global": "left_baseline_spearman_t_global",
    f"mean_left_baseline_signed_overlap_t_top{TOP_K}": left_baseline_overlap_column,
    f"mean_left_baseline_signed_overlap_t_top{TOP_K}_global": f"{left_baseline_overlap_column}_global",
    "mean_right_baseline_spearman_logfc": "right_baseline_spearman_logfc",
    "mean_right_baseline_spearman_logfc_global": "right_baseline_spearman_logfc_global",
    "mean_right_baseline_spearman_t": "right_baseline_spearman_t",
    "mean_right_baseline_spearman_t_global": "right_baseline_spearman_t_global",
    f"mean_right_baseline_signed_overlap_t_top{TOP_K}": right_baseline_overlap_column,
    f"mean_right_baseline_signed_overlap_t_top{TOP_K}_global": f"{right_baseline_overlap_column}_global",
    "mean_left_baseline_peer_count": "left_baseline_peer_count",
    "mean_right_baseline_peer_count": "right_baseline_peer_count",
}

drug_line_time_agg = {
    "n_matched_sample_pairs": ("left_obs_id", "size"),
    "n_matching_conditions": ("matched_condition_key", "nunique"),
}
for target_column, source_column in metric_mean_columns.items():
    drug_line_time_agg[target_column] = (source_column, "mean")
drug_line_time_agg["n_pair_baselines_available"] = (
    "baseline_pair_mean_spearman_logfc",
    lambda values: int(values.notna().sum()),
)
drug_line_time_agg["n_pair_baselines_available_global"] = (
    "baseline_pair_mean_spearman_logfc_global",
    lambda values: int(values.notna().sum()),
)

drug_line_time_metric_summary = (
    matched_pair_metrics.groupby(drug_line_time_group_columns, as_index=False)
    .agg(**drug_line_time_agg)
    .sort_values(drug_line_time_group_columns)
    .reset_index(drop=True)
)
drug_line_time_metric_summary_path = OUTPUT_DIR / "matched_drug_line_time_metric_summary.tsv"
drug_line_time_metric_summary.to_csv(drug_line_time_metric_summary_path, sep="	", index=False)
print(f"Saved drug-line-time metric summary to {drug_line_time_metric_summary_path}")

summary_mean_columns = [
    column_name for column_name in drug_line_time_metric_summary.columns if column_name.startswith("mean_")
]

pair_agg = {
    "n_matched_sample_pairs": ("n_matched_sample_pairs", "sum"),
    "n_matching_lines": ("cell_type", "nunique"),
    "n_matching_drugs": ("pubchem_cid", "nunique"),
    "n_matching_conditions": ("n_matching_conditions", "sum"),
    "n_matching_drug_line_times": ("pubchem_cid", "size"),
}
for column_name in summary_mean_columns:
    pair_agg[column_name] = (column_name, "mean")
pair_agg["n_pair_baselines_available"] = (
    "mean_baseline_pair_spearman_logfc",
    lambda values: int(values.notna().sum()),
)
pair_agg["n_pair_baselines_available_global"] = (
    "mean_baseline_pair_spearman_logfc_global",
    lambda values: int(values.notna().sum()),
)

pair_metric_summary = (
    drug_line_time_metric_summary.groupby(["dataset_a", "dataset_b"], as_index=False)
    .agg(**pair_agg)
    .sort_values(["dataset_a", "dataset_b"])
    .reset_index(drop=True)
)

line_agg = {
    "n_matched_sample_pairs": ("n_matched_sample_pairs", "sum"),
    "n_matching_drugs": ("pubchem_cid", "nunique"),
    "n_matching_conditions": ("n_matching_conditions", "sum"),
    "n_matching_drug_line_times": ("pubchem_cid", "size"),
}
for column_name in summary_mean_columns:
    line_agg[column_name] = (column_name, "mean")
line_agg["n_pair_baselines_available"] = (
    "mean_baseline_pair_spearman_logfc",
    lambda values: int(values.notna().sum()),
)
line_agg["n_pair_baselines_available_global"] = (
    "mean_baseline_pair_spearman_logfc_global",
    lambda values: int(values.notna().sum()),
)

line_metric_summary = (
    drug_line_time_metric_summary.groupby(["dataset_a", "dataset_b", "cell_type"], as_index=False)
    .agg(**line_agg)
    .sort_values(["dataset_a", "dataset_b", "cell_type"])
    .reset_index(drop=True)
)

pair_metric_summary_path = OUTPUT_DIR / "dataset_pair_metric_summary.tsv"
line_metric_summary_path = OUTPUT_DIR / "dataset_pair_line_metric_summary.tsv"
pair_metric_summary.to_csv(pair_metric_summary_path, sep="	", index=False)
line_metric_summary.to_csv(line_metric_summary_path, sep="	", index=False)
print(f"Saved dataset-pair summary to {pair_metric_summary_path}")
print(f"Saved dataset-pair-line summary to {line_metric_summary_path}")

pair_metric_summary_display = pair_metric_summary.copy()
pair_metric_summary_display["dataset_a"] = pair_metric_summary_display["dataset_a"].map(pretty_label)
pair_metric_summary_display["dataset_b"] = pair_metric_summary_display["dataset_b"].map(pretty_label)
display(pair_metric_summary_display)

line_metric_summary_display = line_metric_summary.copy()
line_metric_summary_display["dataset_a"] = line_metric_summary_display["dataset_a"].map(pretty_label)
line_metric_summary_display["dataset_b"] = line_metric_summary_display["dataset_b"].map(pretty_label)
display(line_metric_summary_display)


In [ ]:
display(pair_metric_summary.pivot(index="dataset_a", columns="dataset_b", values="mean_observed_spearman_logfc"))
display(pair_metric_summary.pivot(index="dataset_a", columns="dataset_b", values="mean_observed_spearman_logfc_global"))


In [ ]:
display(pair_metric_summary.pivot(index="dataset_a", columns="dataset_b", values="mean_observed_spearman_t"))
display(pair_metric_summary.pivot(index="dataset_a", columns="dataset_b", values="mean_observed_spearman_t_global"))


In [ ]:
display(pair_metric_summary.pivot(index="dataset_a", columns="dataset_b", values=f"mean_observed_signed_overlap_t_top{TOP_K}"))
display(pair_metric_summary.pivot(index="dataset_a", columns="dataset_b", values=f"mean_observed_signed_overlap_t_top{TOP_K}_global"))


Baseline-adjusted final scores:

For each matched sample pair, these scores subtract the within-dataset same-line / same-time / same-dose other-compound baseline from the observed cross-dataset score. The subtraction is done separately for dataset A and dataset B on the same shared-gene subset used for the observed cross-dataset score, then averaged across the two sides.


In [ ]:
drug_line_time_group_columns = ["dataset_a", "dataset_b", "cell_type", "time_key", "pubchem_cid"]
baseline_adjusted_metrics = matched_pair_metrics.copy()

metric_names = [
    "spearman_logfc",
    "spearman_t",
    f"signed_overlap_t_top{TOP_K}",
    "spearman_logfc_global",
    "spearman_t_global",
    f"signed_overlap_t_top{TOP_K}_global",
]

for metric_name in metric_names:
    observed_col = f"observed_{metric_name}"
    left_baseline_col = f"left_baseline_{metric_name}"
    right_baseline_col = f"right_baseline_{metric_name}"

    left_delta_col = f"delta_vs_left_baseline_{metric_name}"
    right_delta_col = f"delta_vs_right_baseline_{metric_name}"
    pair_mean_delta_col = f"delta_pair_mean_{metric_name}"

    baseline_adjusted_metrics[left_delta_col] = (
        baseline_adjusted_metrics[observed_col] - baseline_adjusted_metrics[left_baseline_col]
    )
    baseline_adjusted_metrics[right_delta_col] = (
        baseline_adjusted_metrics[observed_col] - baseline_adjusted_metrics[right_baseline_col]
    )
    baseline_adjusted_metrics[pair_mean_delta_col] = baseline_adjusted_metrics[
        [left_delta_col, right_delta_col]
    ].mean(axis=1)

baseline_adjusted_metrics_path = OUTPUT_DIR / "matched_sample_pair_metrics_baseline_adjusted.tsv"
baseline_adjusted_metrics.to_csv(baseline_adjusted_metrics_path, sep="	", index=False)
print(f"Saved baseline-adjusted matched-pair metrics to {baseline_adjusted_metrics_path}")

mean_delta_columns = {
    "mean_delta_pair_mean_spearman_logfc": "delta_pair_mean_spearman_logfc",
    "mean_delta_pair_mean_spearman_t": "delta_pair_mean_spearman_t",
    f"mean_delta_pair_mean_signed_overlap_t_top{TOP_K}": f"delta_pair_mean_signed_overlap_t_top{TOP_K}",
    "mean_delta_pair_mean_spearman_logfc_global": "delta_pair_mean_spearman_logfc_global",
    "mean_delta_pair_mean_spearman_t_global": "delta_pair_mean_spearman_t_global",
    f"mean_delta_pair_mean_signed_overlap_t_top{TOP_K}_global": f"delta_pair_mean_signed_overlap_t_top{TOP_K}_global",
    "mean_delta_vs_left_baseline_spearman_logfc": "delta_vs_left_baseline_spearman_logfc",
    "mean_delta_vs_left_baseline_spearman_t": "delta_vs_left_baseline_spearman_t",
    f"mean_delta_vs_left_baseline_signed_overlap_t_top{TOP_K}": f"delta_vs_left_baseline_signed_overlap_t_top{TOP_K}",
    "mean_delta_vs_left_baseline_spearman_logfc_global": "delta_vs_left_baseline_spearman_logfc_global",
    "mean_delta_vs_left_baseline_spearman_t_global": "delta_vs_left_baseline_spearman_t_global",
    f"mean_delta_vs_left_baseline_signed_overlap_t_top{TOP_K}_global": f"delta_vs_left_baseline_signed_overlap_t_top{TOP_K}_global",
    "mean_delta_vs_right_baseline_spearman_logfc": "delta_vs_right_baseline_spearman_logfc",
    "mean_delta_vs_right_baseline_spearman_t": "delta_vs_right_baseline_spearman_t",
    f"mean_delta_vs_right_baseline_signed_overlap_t_top{TOP_K}": f"delta_vs_right_baseline_signed_overlap_t_top{TOP_K}",
    "mean_delta_vs_right_baseline_spearman_logfc_global": "delta_vs_right_baseline_spearman_logfc_global",
    "mean_delta_vs_right_baseline_spearman_t_global": "delta_vs_right_baseline_spearman_t_global",
    f"mean_delta_vs_right_baseline_signed_overlap_t_top{TOP_K}_global": f"delta_vs_right_baseline_signed_overlap_t_top{TOP_K}_global",
}

drug_line_time_delta_agg = {
    "n_matched_sample_pairs": ("left_obs_id", "size"),
    "n_matching_conditions": ("matched_condition_key", "nunique"),
}
for target_column, source_column in mean_delta_columns.items():
    drug_line_time_delta_agg[target_column] = (source_column, "mean")
drug_line_time_delta_agg["n_pair_mean_deltas_available"] = (
    "delta_pair_mean_spearman_logfc",
    lambda values: int(values.notna().sum()),
)
drug_line_time_delta_agg["n_pair_mean_deltas_available_global"] = (
    "delta_pair_mean_spearman_logfc_global",
    lambda values: int(values.notna().sum()),
)

drug_line_time_metric_baseline_adjusted_summary = (
    baseline_adjusted_metrics.groupby(drug_line_time_group_columns, as_index=False)
    .agg(**drug_line_time_delta_agg)
    .sort_values(drug_line_time_group_columns)
    .reset_index(drop=True)
)
drug_line_time_metric_baseline_adjusted_summary_path = OUTPUT_DIR / "matched_drug_line_time_metric_baseline_adjusted_summary.tsv"
drug_line_time_metric_baseline_adjusted_summary.to_csv(drug_line_time_metric_baseline_adjusted_summary_path, sep="	", index=False)
print(f"Saved drug-line-time baseline-adjusted summary to {drug_line_time_metric_baseline_adjusted_summary_path}")

summary_mean_delta_columns = [
    column_name
    for column_name in drug_line_time_metric_baseline_adjusted_summary.columns
    if column_name.startswith("mean_delta_")
]

pair_delta_agg = {
    "n_matched_sample_pairs": ("n_matched_sample_pairs", "sum"),
    "n_matching_lines": ("cell_type", "nunique"),
    "n_matching_drugs": ("pubchem_cid", "nunique"),
    "n_matching_conditions": ("n_matching_conditions", "sum"),
    "n_matching_drug_line_times": ("pubchem_cid", "size"),
}
for column_name in summary_mean_delta_columns:
    pair_delta_agg[column_name] = (column_name, "mean")
pair_delta_agg["n_pair_mean_deltas_available"] = (
    "mean_delta_pair_mean_spearman_logfc",
    lambda values: int(values.notna().sum()),
)
pair_delta_agg["n_pair_mean_deltas_available_global"] = (
    "mean_delta_pair_mean_spearman_logfc_global",
    lambda values: int(values.notna().sum()),
)

pair_metric_baseline_adjusted_summary = (
    drug_line_time_metric_baseline_adjusted_summary.groupby(["dataset_a", "dataset_b"], as_index=False)
    .agg(**pair_delta_agg)
    .sort_values(["dataset_a", "dataset_b"])
    .reset_index(drop=True)
)

line_delta_agg = {
    "n_matched_sample_pairs": ("n_matched_sample_pairs", "sum"),
    "n_matching_drugs": ("pubchem_cid", "nunique"),
    "n_matching_conditions": ("n_matching_conditions", "sum"),
    "n_matching_drug_line_times": ("pubchem_cid", "size"),
}
for column_name in summary_mean_delta_columns:
    line_delta_agg[column_name] = (column_name, "mean")
line_delta_agg["n_pair_mean_deltas_available"] = (
    "mean_delta_pair_mean_spearman_logfc",
    lambda values: int(values.notna().sum()),
)
line_delta_agg["n_pair_mean_deltas_available_global"] = (
    "mean_delta_pair_mean_spearman_logfc_global",
    lambda values: int(values.notna().sum()),
)

line_metric_baseline_adjusted_summary = (
    drug_line_time_metric_baseline_adjusted_summary.groupby(["dataset_a", "dataset_b", "cell_type"], as_index=False)
    .agg(**line_delta_agg)
    .sort_values(["dataset_a", "dataset_b", "cell_type"])
    .reset_index(drop=True)
)

pair_metric_baseline_adjusted_summary_path = OUTPUT_DIR / "dataset_pair_metric_baseline_adjusted_summary.tsv"
line_metric_baseline_adjusted_summary_path = OUTPUT_DIR / "dataset_pair_line_metric_baseline_adjusted_summary.tsv"
pair_metric_baseline_adjusted_summary.to_csv(pair_metric_baseline_adjusted_summary_path, sep="	", index=False)
line_metric_baseline_adjusted_summary.to_csv(line_metric_baseline_adjusted_summary_path, sep="	", index=False)
print(f"Saved baseline-adjusted dataset-pair summary to {pair_metric_baseline_adjusted_summary_path}")
print(f"Saved baseline-adjusted dataset-pair-line summary to {line_metric_baseline_adjusted_summary_path}")

pair_metric_baseline_adjusted_summary_display = pair_metric_baseline_adjusted_summary.copy()
pair_metric_baseline_adjusted_summary_display["dataset_a"] = pair_metric_baseline_adjusted_summary_display["dataset_a"].map(pretty_label)
pair_metric_baseline_adjusted_summary_display["dataset_b"] = pair_metric_baseline_adjusted_summary_display["dataset_b"].map(pretty_label)
display(pair_metric_baseline_adjusted_summary_display)

line_metric_baseline_adjusted_summary_display = line_metric_baseline_adjusted_summary.copy()
line_metric_baseline_adjusted_summary_display["dataset_a"] = line_metric_baseline_adjusted_summary_display["dataset_a"].map(pretty_label)
line_metric_baseline_adjusted_summary_display["dataset_b"] = line_metric_baseline_adjusted_summary_display["dataset_b"].map(pretty_label)
display(line_metric_baseline_adjusted_summary_display)


In [ ]:

import sys

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.cluster_bootstrap_ci import cluster_bca_nested_mean_ci_table, summarize_ci_half_width_ranges

BOOTSTRAP_ITERATIONS = 2000
BOOTSTRAP_RANDOM_SEED = 20260505

signature_ci_metrics = {
    "observed_spearman_logfc": "mean_observed_spearman_logfc",
    "baseline_pair_spearman_logfc": "mean_baseline_pair_spearman_logfc",
}
signature_delta_ci_metrics = {
    "delta_pair_mean_spearman_logfc": "mean_delta_pair_mean_spearman_logfc",
}

overlap_signature_cluster_bca_ci = pd.concat(
    [
        cluster_bca_nested_mean_ci_table(
            drug_line_time_metric_summary,
            group_cols=["dataset_a", "dataset_b"],
            metric_cols=signature_ci_metrics,
            cluster_col="pubchem_cid",
            n_boot=BOOTSTRAP_ITERATIONS,
            seed=BOOTSTRAP_RANDOM_SEED,
            summary_level="dataset_pair",
        ),
        cluster_bca_nested_mean_ci_table(
            drug_line_time_metric_summary,
            group_cols=["dataset_a", "dataset_b", "cell_type"],
            metric_cols=signature_ci_metrics,
            cluster_col="pubchem_cid",
            n_boot=BOOTSTRAP_ITERATIONS,
            seed=BOOTSTRAP_RANDOM_SEED,
            summary_level="dataset_pair_line",
        ),
        cluster_bca_nested_mean_ci_table(
            drug_line_time_metric_baseline_adjusted_summary,
            group_cols=["dataset_a", "dataset_b"],
            metric_cols=signature_delta_ci_metrics,
            cluster_col="pubchem_cid",
            n_boot=BOOTSTRAP_ITERATIONS,
            seed=BOOTSTRAP_RANDOM_SEED,
            summary_level="dataset_pair_baseline_adjusted",
        ),
        cluster_bca_nested_mean_ci_table(
            drug_line_time_metric_baseline_adjusted_summary,
            group_cols=["dataset_a", "dataset_b", "cell_type"],
            metric_cols=signature_delta_ci_metrics,
            cluster_col="pubchem_cid",
            n_boot=BOOTSTRAP_ITERATIONS,
            seed=BOOTSTRAP_RANDOM_SEED,
            summary_level="dataset_pair_line_baseline_adjusted",
        ),
    ],
    ignore_index=True,
)
overlap_signature_cluster_bca_ci_path = OUTPUT_DIR / "signature_similarity_cluster_bca_ci.tsv"
overlap_signature_cluster_bca_ci.to_csv(overlap_signature_cluster_bca_ci_path, sep="\t", index=False)
print(f"Saved overlap signature cluster-BCa CI table to {overlap_signature_cluster_bca_ci_path}")

overlap_signature_cluster_bca_ci_ranges = summarize_ci_half_width_ranges(overlap_signature_cluster_bca_ci)
overlap_signature_cluster_bca_ci_ranges_path = OUTPUT_DIR / "signature_similarity_cluster_bca_ci_ranges.tsv"
overlap_signature_cluster_bca_ci_ranges.to_csv(overlap_signature_cluster_bca_ci_ranges_path, sep="\t", index=False)
print(f"Saved overlap signature CI half-width ranges to {overlap_signature_cluster_bca_ci_ranges_path}")
display(overlap_signature_cluster_bca_ci)
display(overlap_signature_cluster_bca_ci_ranges)


In [ ]:
display(pair_metric_baseline_adjusted_summary.pivot(index="dataset_a", columns="dataset_b", values="mean_delta_pair_mean_spearman_logfc"))
display(pair_metric_baseline_adjusted_summary.pivot(index="dataset_a", columns="dataset_b", values="mean_delta_pair_mean_spearman_logfc_global"))


In [ ]:
display(pair_metric_baseline_adjusted_summary.pivot(index="dataset_a", columns="dataset_b", values="mean_delta_pair_mean_spearman_t"))
display(pair_metric_baseline_adjusted_summary.pivot(index="dataset_a", columns="dataset_b", values="mean_delta_pair_mean_spearman_t_global"))


In [ ]:
display(pair_metric_baseline_adjusted_summary.pivot(index="dataset_a", columns="dataset_b", values=f"mean_delta_pair_mean_signed_overlap_t_top{TOP_K}"))
display(pair_metric_baseline_adjusted_summary.pivot(index="dataset_a", columns="dataset_b", values=f"mean_delta_pair_mean_signed_overlap_t_top{TOP_K}_global"))


Baseline score summaries:

These tables show the within-dataset baseline scores themselves, using the pair-mean baseline already defined above as the mean of the dataset A and dataset B same-line / same-time / same-dose other-compound baselines.


In [ ]:
pair_metric_baseline_summary = pair_metric_summary[
    [
        "dataset_a",
        "dataset_b",
        "n_matched_sample_pairs",
        "n_matching_lines",
        "n_matching_drugs",
        "n_matching_conditions",
        "n_matching_drug_line_times",
        "mean_baseline_pair_spearman_logfc",
        "mean_baseline_pair_spearman_logfc_global",
        "mean_baseline_pair_spearman_t",
        "mean_baseline_pair_spearman_t_global",
        f"mean_baseline_pair_signed_overlap_t_top{TOP_K}",
        f"mean_baseline_pair_signed_overlap_t_top{TOP_K}_global",
        "mean_left_baseline_spearman_logfc",
        "mean_left_baseline_spearman_logfc_global",
        "mean_left_baseline_spearman_t",
        "mean_left_baseline_spearman_t_global",
        f"mean_left_baseline_signed_overlap_t_top{TOP_K}",
        f"mean_left_baseline_signed_overlap_t_top{TOP_K}_global",
        "mean_right_baseline_spearman_logfc",
        "mean_right_baseline_spearman_logfc_global",
        "mean_right_baseline_spearman_t",
        "mean_right_baseline_spearman_t_global",
        f"mean_right_baseline_signed_overlap_t_top{TOP_K}",
        f"mean_right_baseline_signed_overlap_t_top{TOP_K}_global",
        "n_pair_baselines_available",
        "n_pair_baselines_available_global",
    ]
].copy()

line_metric_baseline_summary = line_metric_summary[
    [
        "dataset_a",
        "dataset_b",
        "cell_type",
        "n_matched_sample_pairs",
        "n_matching_drugs",
        "n_matching_conditions",
        "n_matching_drug_line_times",
        "mean_baseline_pair_spearman_logfc",
        "mean_baseline_pair_spearman_logfc_global",
        "mean_baseline_pair_spearman_t",
        "mean_baseline_pair_spearman_t_global",
        f"mean_baseline_pair_signed_overlap_t_top{TOP_K}",
        f"mean_baseline_pair_signed_overlap_t_top{TOP_K}_global",
        "n_pair_baselines_available",
        "n_pair_baselines_available_global",
    ]
].copy()

pair_metric_baseline_summary_display = pair_metric_baseline_summary.copy()
pair_metric_baseline_summary_display["dataset_a"] = pair_metric_baseline_summary_display["dataset_a"].map(pretty_label)
pair_metric_baseline_summary_display["dataset_b"] = pair_metric_baseline_summary_display["dataset_b"].map(pretty_label)
display(pair_metric_baseline_summary_display)

line_metric_baseline_summary_display = line_metric_baseline_summary.copy()
line_metric_baseline_summary_display["dataset_a"] = line_metric_baseline_summary_display["dataset_a"].map(pretty_label)
line_metric_baseline_summary_display["dataset_b"] = line_metric_baseline_summary_display["dataset_b"].map(pretty_label)
display(line_metric_baseline_summary_display)


In [ ]:
display(pair_metric_baseline_summary.pivot(index="dataset_a", columns="dataset_b", values="mean_baseline_pair_spearman_logfc"))
display(pair_metric_baseline_summary.pivot(index="dataset_a", columns="dataset_b", values="mean_baseline_pair_spearman_logfc_global"))


In [ ]:
display(pair_metric_baseline_summary.pivot(index="dataset_a", columns="dataset_b", values="mean_baseline_pair_spearman_t"))
display(pair_metric_baseline_summary.pivot(index="dataset_a", columns="dataset_b", values="mean_baseline_pair_spearman_t_global"))


In [ ]:
display(pair_metric_baseline_summary.pivot(index="dataset_a", columns="dataset_b", values=f"mean_baseline_pair_signed_overlap_t_top{TOP_K}"))
display(pair_metric_baseline_summary.pivot(index="dataset_a", columns="dataset_b", values=f"mean_baseline_pair_signed_overlap_t_top{TOP_K}_global"))


**T Strength vs Correlation**

This section asks whether matched samples with larger moderated `t` magnitude also show higher cross-dataset similarity.

For each matched sample pair, it computes:
- `left_mean_abs_t`: mean absolute moderated `t` for the left sample on the same pairwise shared-gene set used for the observed cross-dataset score
- `right_mean_abs_t`: same for the right sample
- `pair_mean_abs_t`: average of the left and right values

The summary tables still report dataset-pair-level Spearman relationships. The scatter plots, however, now use one point per exact matched condition, i.e.
`pubchem_cid + cell_type + time_key + left_dose_key + right_dose_key`,
so multiple matched dose pairs for the same drug-line-time appear as separate points.

The same analysis is repeated on the line-specific global shared-gene set using the `*_global` columns.

In the scatter plots, the x-axis uses `log10(mean absolute moderated t)` to reduce domination by extreme-`|t|` outliers.


In [ ]:
def safe_column_spearman(frame: pd.DataFrame, x_col: str, y_col: str) -> float:
    subset = frame[[x_col, y_col]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(subset) < 2 or subset[x_col].nunique() < 2 or subset[y_col].nunique() < 2:
        return float("nan")
    return float(subset[x_col].corr(subset[y_col], method="spearman"))


t_strength_pair_relationship_summary = pd.DataFrame(
    [
        {
            "dataset_a": dataset_a,
            "dataset_b": dataset_b,
            "n_matched_sample_pairs": int(len(group)),
            "spearman_pair_mean_abs_t_vs_observed_logfc": safe_column_spearman(group, "pair_mean_abs_t", "observed_spearman_logfc"),
            "spearman_pair_mean_abs_t_vs_observed_t": safe_column_spearman(group, "pair_mean_abs_t", "observed_spearman_t"),
            f"spearman_pair_mean_abs_t_vs_observed_signed_overlap_t_top{TOP_K}": safe_column_spearman(group, "pair_mean_abs_t", f"observed_signed_overlap_t_top{TOP_K}"),
            "spearman_pair_mean_abs_t_global_vs_observed_logfc_global": safe_column_spearman(group, "pair_mean_abs_t_global", "observed_spearman_logfc_global"),
            "spearman_pair_mean_abs_t_global_vs_observed_t_global": safe_column_spearman(group, "pair_mean_abs_t_global", "observed_spearman_t_global"),
            f"spearman_pair_mean_abs_t_global_vs_observed_signed_overlap_t_top{TOP_K}_global": safe_column_spearman(group, "pair_mean_abs_t_global", f"observed_signed_overlap_t_top{TOP_K}_global"),
        }
        for (dataset_a, dataset_b), group in matched_pair_metrics.groupby(["dataset_a", "dataset_b"], sort=False)
    ]
).sort_values(["dataset_a", "dataset_b"]).reset_index(drop=True)

matched_condition_group_columns = [
    "dataset_a",
    "dataset_b",
    "cell_type",
    "time_key",
    "pubchem_cid",
    "left_dose_key",
    "right_dose_key",
    "matched_condition_key",
]

matched_condition_t_strength_summary = (
    matched_pair_metrics.groupby(matched_condition_group_columns, as_index=False)
    .agg(
        n_matched_sample_pairs=("left_obs_id", "size"),
        mean_pair_mean_abs_t=("pair_mean_abs_t", "mean"),
        mean_pair_mean_abs_t_global=("pair_mean_abs_t_global", "mean"),
        mean_observed_spearman_logfc=("observed_spearman_logfc", "mean"),
        mean_observed_spearman_t=("observed_spearman_t", "mean"),
        **{f"mean_observed_signed_overlap_t_top{TOP_K}": (f"observed_signed_overlap_t_top{TOP_K}", "mean")},
        mean_observed_spearman_logfc_global=("observed_spearman_logfc_global", "mean"),
        mean_observed_spearman_t_global=("observed_spearman_t_global", "mean"),
        **{f"mean_observed_signed_overlap_t_top{TOP_K}_global": (f"observed_signed_overlap_t_top{TOP_K}_global", "mean")},
    )
    .sort_values(matched_condition_group_columns)
    .reset_index(drop=True)
)


t_strength_matched_condition_relationship_summary = pd.DataFrame(
    [
        {
            "dataset_a": dataset_a,
            "dataset_b": dataset_b,
            "n_matching_condition_dose_pairs": int(len(group)),
            "spearman_mean_abs_t_vs_observed_logfc": safe_column_spearman(group, "mean_pair_mean_abs_t", "mean_observed_spearman_logfc"),
            "spearman_mean_abs_t_vs_observed_t": safe_column_spearman(group, "mean_pair_mean_abs_t", "mean_observed_spearman_t"),
            f"spearman_mean_abs_t_vs_observed_signed_overlap_t_top{TOP_K}": safe_column_spearman(group, "mean_pair_mean_abs_t", f"mean_observed_signed_overlap_t_top{TOP_K}"),
            "spearman_mean_abs_t_global_vs_observed_logfc_global": safe_column_spearman(group, "mean_pair_mean_abs_t_global", "mean_observed_spearman_logfc_global"),
            "spearman_mean_abs_t_global_vs_observed_t_global": safe_column_spearman(group, "mean_pair_mean_abs_t_global", "mean_observed_spearman_t_global"),
            f"spearman_mean_abs_t_global_vs_observed_signed_overlap_t_top{TOP_K}_global": safe_column_spearman(group, "mean_pair_mean_abs_t_global", f"mean_observed_signed_overlap_t_top{TOP_K}_global"),
        }
        for (dataset_a, dataset_b), group in matched_condition_t_strength_summary.groupby(["dataset_a", "dataset_b"], sort=False)
    ]
).sort_values(["dataset_a", "dataset_b"]).reset_index(drop=True)

for frame in [t_strength_pair_relationship_summary, t_strength_matched_condition_relationship_summary]:
    frame["dataset_a"] = frame["dataset_a"].map(pretty_label)
    frame["dataset_b"] = frame["dataset_b"].map(pretty_label)

display(t_strength_pair_relationship_summary)
display(t_strength_matched_condition_relationship_summary)


In [ ]:
plot_frame = matched_condition_t_strength_summary.copy()
plot_frame = plot_frame.loc[
    np.isfinite(plot_frame["mean_pair_mean_abs_t"].to_numpy(dtype=float))
    & (plot_frame["mean_pair_mean_abs_t"].to_numpy(dtype=float) > 0)
].copy()
plot_frame["log10_mean_pair_mean_abs_t"] = np.log10(plot_frame["mean_pair_mean_abs_t"].to_numpy(dtype=float))
plot_frame["dataset_pair"] = plot_frame["dataset_a"].map(pretty_label) + " vs " + plot_frame["dataset_b"].map(pretty_label)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

sns.scatterplot(
    data=plot_frame,
    x="log10_mean_pair_mean_abs_t",
    y="mean_observed_spearman_logfc",
    hue="dataset_pair",
    alpha=0.7,
    s=18,
    ax=axes[0],
)
sns.regplot(
    data=plot_frame,
    x="log10_mean_pair_mean_abs_t",
    y="mean_observed_spearman_logfc",
    scatter=False,
    color="black",
    line_kws={"linewidth": 2.0, "linestyle": "--"},
    ax=axes[0],
)
logfc_rho = safe_column_spearman(plot_frame, "log10_mean_pair_mean_abs_t", "mean_observed_spearman_logfc")
logfc_subset = plot_frame[["log10_mean_pair_mean_abs_t", "mean_observed_spearman_logfc"]].replace([np.inf, -np.inf], np.nan).dropna()
axes[0].text(
    0.03,
    0.97,
    f"Spearman rho = {logfc_rho:.3f}\nn = {len(logfc_subset):,}",
    transform=axes[0].transAxes,
    ha="left",
    va="top",
    fontsize=10,
    bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "alpha": 0.85, "edgecolor": "0.8"},
)
axes[0].set_title("Matched condition-dose log mean |t| vs observed logFC correlation")
axes[0].set_xlabel("log10(mean absolute moderated t across datasets)")
axes[0].set_ylabel("Observed cross-dataset Spearman(logFC)")

sns.scatterplot(
    data=plot_frame,
    x="log10_mean_pair_mean_abs_t",
    y="mean_observed_spearman_t",
    hue="dataset_pair",
    alpha=0.7,
    s=18,
    ax=axes[1],
)
sns.regplot(
    data=plot_frame,
    x="log10_mean_pair_mean_abs_t",
    y="mean_observed_spearman_t",
    scatter=False,
    color="black",
    line_kws={"linewidth": 2.0, "linestyle": "--"},
    ax=axes[1],
)
t_rho = safe_column_spearman(plot_frame, "log10_mean_pair_mean_abs_t", "mean_observed_spearman_t")
t_subset = plot_frame[["log10_mean_pair_mean_abs_t", "mean_observed_spearman_t"]].replace([np.inf, -np.inf], np.nan).dropna()
axes[1].text(
    0.03,
    0.97,
    f"Spearman rho = {t_rho:.3f}\nn = {len(t_subset):,}",
    transform=axes[1].transAxes,
    ha="left",
    va="top",
    fontsize=10,
    bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "alpha": 0.85, "edgecolor": "0.8"},
)
axes[1].set_title("Matched condition-dose log mean |t| vs observed t correlation")
axes[1].set_xlabel("log10(mean absolute moderated t across datasets)")
axes[1].set_ylabel("Observed cross-dataset Spearman(t)")

handles, labels = axes[1].get_legend_handles_labels()
if axes[0].legend_ is not None:
    axes[0].legend_.remove()
axes[1].legend(handles=handles, labels=labels, title="Dataset pair", bbox_to_anchor=(1.02, 1), loc="upper left")

t_strength_scatter_path = OUTPUT_DIR / "matched_condition_log10_mean_abs_t_vs_observed_spearman_scatter.pdf"
fig.savefig(t_strength_scatter_path, bbox_inches="tight")
print(f"Saved plot to {t_strength_scatter_path}")

t_strength_scatter_description_path = OUTPUT_DIR / "matched_condition_log10_mean_abs_t_vs_observed_spearman_scatter_description.txt"
description_lines = [
    "Plot: matched-condition perturbation strength vs cross-dataset signature similarity.",
    "",
    f"Source table: {matched_pair_metrics_path}",
    f"Output plot: {t_strength_scatter_path}",
    f"Number of plotted matched condition-dose points: {len(plot_frame):,}",
    "",
    "Generation steps:",
    "1. Start from matched_sample_pair_metrics.tsv, which contains precomputed pairwise signature-similarity metrics for matched sample pairs across datasets.",
    "2. Aggregate sample-pair rows to matched condition-dose rows grouped by dataset_a, dataset_b, cell_type, time_key, pubchem_cid, left_dose_key, right_dose_key, and matched_condition_key.",
    "3. For each matched condition-dose row, average pair_mean_abs_t, observed_spearman_logfc, and observed_spearman_t over the available matched sample pairs.",
    "4. Keep rows with finite, positive mean_pair_mean_abs_t and plot log10(mean_pair_mean_abs_t) on the x-axis.",
    "5. Draw two scatter panels: observed cross-dataset Spearman(logFC) on the left and observed cross-dataset Spearman(moderated t) on the right.",
    "6. Color points by dataset pair and overlay a dashed black linear regression trend line for visual orientation.",
    "7. Report Spearman rho in each panel using the plotted matched condition-dose points, after dropping non-finite x/y values.",
    "",
    "What the plot shows:",
    f"- Left panel: association between log10 mean absolute moderated t and observed cross-dataset Spearman(logFC); Spearman rho = {logfc_rho:.3f}, n = {len(logfc_subset):,}.",
    f"- Right panel: association between log10 mean absolute moderated t and observed cross-dataset Spearman(moderated t); Spearman rho = {t_rho:.3f}, n = {len(t_subset):,}.",
    "- Higher mean absolute moderated t corresponds to stronger perturbation signal across the two datasets for a matched compound, cell type, time, and dose pair.",
    "- The positive Spearman correlations indicate that stronger perturbation signals tend to be more reproducible across datasets, but the scatter remains broad, so t-strength is only a partial explanation of cross-dataset similarity.",
    "- The figure is descriptive and uses already precomputed signature-similarity outputs; it does not recompute signatures or add uncertainty intervals.",
]
t_strength_scatter_description_path.write_text("\n".join(description_lines) + "\n")
print(f"Saved plot description to {t_strength_scatter_description_path}")
plt.show()


Optional cleanup:

```python
close_all_line_sources()
```


**Dataset-Pair Heatmaps**

These heatmaps summarize the dataset-pair means for `Spearman(logFC)` at three levels:
- observed cross-dataset score
- pair baseline score
- observed minus baseline pair score

They use the same dataset labels and lower-triangular layout as [plot_overlap_heatmaps.ipynb](./plot_overlap_heatmaps.ipynb). The diagonal is masked because these are cross-dataset summaries only.

In [ ]:
HEATMAP_OUTPUT_PATHS = {
    "mean_observed_spearman_logfc": OUTPUT_DIR / "dataset_pair_mean_observed_spearman_logfc_heatmap.pdf",
    "mean_baseline_pair_spearman_logfc": OUTPUT_DIR / "dataset_pair_mean_baseline_pair_spearman_logfc_heatmap.pdf",
    "mean_delta_pair_mean_spearman_logfc": OUTPUT_DIR / "dataset_pair_mean_delta_pair_mean_spearman_logfc_heatmap.pdf",
}


def build_symmetric_pair_metric_matrix(
    summary_frame: pd.DataFrame,
    value_col: str,
    dataset_order: list[str],
) -> pd.DataFrame:
    matrix = pd.DataFrame(np.nan, index=dataset_order, columns=dataset_order, dtype=float)
    for _, row in summary_frame.iterrows():
        dataset_a = str(row["dataset_a"])
        dataset_b = str(row["dataset_b"])
        if dataset_a not in matrix.index or dataset_b not in matrix.columns:
            continue
        value = pd.to_numeric(pd.Series([row[value_col]]), errors="coerce").iloc[0]
        matrix.loc[dataset_a, dataset_b] = value
        matrix.loc[dataset_b, dataset_a] = value
    return matrix


heatmap_dataset_order = [dataset_name for dataset_name in DATASET_ORDER if dataset_name in active_datasets]

observed_baseline_values = pd.concat(
    [
        pair_metric_summary["mean_observed_spearman_logfc"],
        pair_metric_summary["mean_baseline_pair_spearman_logfc"],
    ],
    ignore_index=True,
).replace([np.inf, -np.inf], np.nan)
observed_baseline_values = observed_baseline_values.dropna()
if observed_baseline_values.empty:
    observed_baseline_vmin = None
    observed_baseline_vmax = None
else:
    observed_baseline_vmin = float(observed_baseline_values.min())
    observed_baseline_vmax = float(observed_baseline_values.max())


delta_values = (
    pair_metric_baseline_adjusted_summary["mean_delta_pair_mean_spearman_logfc"]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)
if delta_values.empty:
    delta_abs_max = None
else:
    delta_abs_max = float(np.nanmax(np.abs(delta_values.to_numpy(dtype=float))))



def plot_pair_metric_heatmap(
    summary_frame: pd.DataFrame,
    value_col: str,
    title: str,
    cmap: str,
    output_path: Path,
    *,
    dataset_order: Optional[list[str]] = None,
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
    center: Optional[float] = None,
    cbar_label: Optional[str] = None,
):
    dataset_order = dataset_order or heatmap_dataset_order
    matrix = build_symmetric_pair_metric_matrix(summary_frame, value_col, dataset_order)
    display_matrix = matrix.rename(index=pretty_label, columns=pretty_label).iloc[1:, :-1]
    mask = np.triu(np.ones(display_matrix.shape, dtype=bool), k=1)

    fig_width = max(5.5, 1.35 * len(display_matrix.columns))
    fig_height = max(4.5, 1.15 * len(display_matrix.index))
    fig, ax = plt.subplots(figsize=(fig_width, fig_height), constrained_layout=True)
    sns.heatmap(
        display_matrix,
        mask=mask,
        annot=True,
        fmt=".3f",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        center=center,
        linewidths=0.5,
        linecolor="white",
        square=True,
        cbar_kws={"shrink": 0.85},
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
    ax.grid(False)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, bbox_inches="tight")
    plt.show()
    print(f"Saved heatmap to {output_path}")


print("Heatmap datasets:", ", ".join(pretty_label(dataset_name) for dataset_name in heatmap_dataset_order))


In [ ]:
plot_pair_metric_heatmap(
    pair_metric_summary,
    value_col="mean_observed_spearman_logfc",
    title="Cross-Dataset Spearman(logFC)",
    cmap="YlGnBu",
    output_path=HEATMAP_OUTPUT_PATHS["mean_observed_spearman_logfc"],
    dataset_order=heatmap_dataset_order,
    vmin=observed_baseline_vmin,
    vmax=observed_baseline_vmax,
    cbar_label="Mean observed Spearman(logFC)",
)


In [ ]:
plot_pair_metric_heatmap(
    pair_metric_summary,
    value_col="mean_baseline_pair_spearman_logfc",
    title="Baseline Pair Spearman(logFC)",
    cmap="YlOrBr",
    output_path=HEATMAP_OUTPUT_PATHS["mean_baseline_pair_spearman_logfc"],
    dataset_order=heatmap_dataset_order,
    vmin=observed_baseline_vmin,
    vmax=observed_baseline_vmax,
    cbar_label="Mean baseline pair Spearman(logFC)",
)


In [ ]:
plot_pair_metric_heatmap(
    pair_metric_baseline_adjusted_summary,
    value_col="mean_delta_pair_mean_spearman_logfc",
    title="Observed Minus Baseline Pair Spearman(logFC)",
    cmap="RdBu_r",
    output_path=HEATMAP_OUTPUT_PATHS["mean_delta_pair_mean_spearman_logfc"],
    dataset_order=heatmap_dataset_order,
    vmin=None if delta_abs_max is None else -delta_abs_max,
    vmax=None if delta_abs_max is None else delta_abs_max,
    center=0.0,
    cbar_label="Mean delta Spearman(logFC)",
)
